# Similis — Inspeção Interativa no Databricks

Notebook para rodar o pipeline Similis dentro do **Databricks**, lendo dados diretamente das tabelas Delta (mesmas do job `similis_farma` definido em `resources/similis.yml`) e gravando o resultado em `groceries_ops.similis.recommendations`.

## Como executar

1. Suba o repo para o workspace via `databricks bundle deploy --target dev` (ou abra como Databricks Repo).
2. Abra este notebook num cluster Spark 15.4 LTS (Python 3.10) com permissão de leitura nas tabelas e Volume citados.
3. Execute célula a célula. A primeira execução de cada subcategoria leva ~10–90 min (BGE-M3 em CPU). Cache no Volume torna as próximas execuções rápidas.

---
## 1. Setup — instala libs e ajusta `sys.path`

Instala as dependências do Similis no kernel atual (mesmas declaradas em `resources/similis.yml`) e adiciona `src/similis/` ao `sys.path` para importar `parts/*`.

In [0]:
%pip install -q \
  "numpy==1.24.4" \
  "pandas==1.5.3" \
  "scipy==1.11.4" \
  "huggingface_hub==0.25.2" \
  "transformers==4.46.3" \
  "sentence-transformers==3.3.1" \
  "torch==2.3.1" \
  "scikit-learn==1.3.2" \
  "faiss-cpu" "unidecode" "pyyaml" "pyarrow"

dbutils.library.restartPython() 

In [0]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Descobre src/similis/ a partir do diretório do notebook (funciona em Databricks
# Repos e em Workspace files quando o repo é deployado via DAB).
NB_DIR = Path.cwd()
SIMILIS_ROOT = Path('/Workspace/Users/<seu.usuario>/<repo>/files/src/similis/')
for parent in [NB_DIR, *NB_DIR.parents]:
    if (parent / "main.py").exists() and (parent / "parts").is_dir():
        SIMILIS_ROOT = parent
        break
if SIMILIS_ROOT is None:
    for parent in NB_DIR.parents:
        cand = parent / "src" / "similis"
        if (cand / "main.py").exists() and (cand / "parts").is_dir():
            SIMILIS_ROOT = cand
            break

assert SIMILIS_ROOT is not None, f"Não achei src/similis/ a partir de {NB_DIR}"
sys.path.insert(0, str(SIMILIS_ROOT))

from parts.config_loader import ConfigLoader
from parts.constants import (
    EXCLUDED_SUBCATEGORIES,
    FARMA_SUBCATEGORIES_DEFAULT,
    SUBCATEGORY_NAMES,
    category_slug,
    nested_subcategoria,
)
from parts.data_loader import load_catalog
from parts.embedder import BgeM3Embedder
from parts.normalizer import normalize_dataframe
from parts.ranker import recommend
from parts.writer import write_recommendations

INFOS_PATH = str(SIMILIS_ROOT / "infos.yaml")

print(f"SIMILIS_ROOT : {SIMILIS_ROOT}")
print(f"INFOS_PATH   : {INFOS_PATH}")
print("Imports OK")
print()
print("Subcategorias Farma reconhecidas (FARMA_SUBCATEGORIES_DEFAULT):")
for slug in FARMA_SUBCATEGORIES_DEFAULT:
    print(f"  {slug:22s} -> {SUBCATEGORY_NAMES.get(slug, '?')}")

---
## 2. Escolher subcategoria e carregar a config

`SUBCATEGORY_SLUG` é um dos slugs em `parts.constants.FARMA_SUBCATEGORIES_DEFAULT`. O `ConfigLoader` tenta ler de `groceries_ops.similis.config_subcategoria` (Delta) primeiro; se a tabela estiver vazia ou indisponível, faz fallback para a entrada correspondente em `infos.yaml`.

In [0]:
SUBCATEGORY_SLUG = "fraldas__troca_de_fralda"   # slug = subcategoria__categoria; ex.: "", "lencos_umedecidos__troca_de_fralda", "talco__troca_de_fralda"

assert SUBCATEGORY_SLUG.lower() not in EXCLUDED_SUBCATEGORIES, (
    f"{SUBCATEGORY_SLUG!r} está em EXCLUDED_SUBCATEGORIES — Similis Farma não cobre medicamentos."
)

loader = ConfigLoader(spark, INFOS_PATH)
config = loader.get(SUBCATEGORY_SLUG)

print(f"slug              : {SUBCATEGORY_SLUG}")
print(f"subcategory_name  : {config.subcategory_name}")
print(f"source            : {config.source}")
print(f"top_k             : {config.top_k}")
print(f"min_score         : {config.min_score}")
print(f"# regras          : {len(config.rules)}")
print(f"hard_filter       : {[r.attribute for r in config.hard_filter_rules]}")
print(f"soft_boost        : {[(r.attribute, r.boost_factor) for r in config.soft_boost_rules]}")
print(f"config_hash       : {config.config_hash}")

---
## 3. Carregar catálogo Farma do Spark, normalizar e gerar embeddings

`load_catalog` (em `parts.data_loader`) faz exatamente o que `get_universe_skus` + pivot do `skus_metadata` definem para o universo Farma:

- `business_categorization.pharmacy.business_type_name = 'Farmácia'`
- `department_name` ∉ {Medicamentos, Itens de Bloqueio, Teste Depto Shop}
- `category_id` na allow-list de UUIDs
- `subcategory_name` igual ao da config carregada

Devolve um pandas DataFrame com 1 linha por EAN e uma coluna por atributo configurado.

`BgeM3Embedder` cacheia os vetores em parquet no Volume `/Volumes/groceries_ops/similis/embeddings_cache/<slug>.parquet` — execuções seguintes só recomputam EANs novos ou cujo `text_canon` mudou. Se o cache foi gerado por modelo de outra dimensão (ex.: troca de embedding), é descartado automaticamente.

> Tempo: na 1ª execução de uma subcategoria pode levar 10–90 min em CPU dependendo do número de EANs. Releituras são em segundos.

In [0]:
# Disable Arrow optimization — numpy 1.24 is incompatible with the system pyarrow/pandas on DBR 13.3
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

print("Carregando catálogo do Spark...")
t0 = time.time()
df = load_catalog(spark, config.subcategory_name, config)
t_load = time.time() - t0
print(f"  load_catalog: {t_load:.1f}s, {len(df)} EANs únicos")

if df.empty:
    raise RuntimeError(f"Catálogo vazio para {config.subcategory_name!r}.")

print("\nNormalizando text_canon...")
t0 = time.time()
df_norm = normalize_dataframe(df, config)
t_norm = time.time() - t0
canon_len = df_norm["text_canon"].str.len()
print(f"  normalize_dataframe: {t_norm:.1f}s")
print(f"  text_canon — média={canon_len.mean():.0f} chars  mediana={canon_len.median():.0f}  máx={canon_len.max()}")

# --- TESTE: subamostra para iteração rápida (zere TEST_SAMPLE para rodar completo) ---
TEST_SAMPLE = 0
if TEST_SAMPLE and len(df_norm) > TEST_SAMPLE:
    df_norm = df_norm.sample(n=TEST_SAMPLE, random_state=42).reset_index(drop=True)
    print(f"  [TESTE] subamostrado para {len(df_norm)} EANs")

print("\nGerando embeddings BGE-M3 (cache em Volume UC)...")
embedder = BgeM3Embedder()
t0 = time.time()
embeddings = embedder.encode_dataframe(df_norm, SUBCATEGORY_SLUG)
t_emb = time.time() - t0
print(f"  encode_dataframe: {t_emb:.1f}s ({t_emb*1000/max(len(df_norm),1):.1f}ms/EAN)")
print(f"  shape: {embeddings.shape}")
print(f"  norma média: {np.linalg.norm(embeddings, axis=1).mean():.4f}  (~1.0 — vetores normalizados)")
print(f"  cache em: {os.path.join(embedder.cache_dir, f'{SUBCATEGORY_SLUG}.parquet')}")

---
## 4. Rodar o ranker e inspecionar qualitativamente

Particiona por `hard_filter` (se houver), busca top-K via FAISS por similaridade cosseno e re-ranqueia com os `soft_boost`. Em seguida, mostra uma amostra aleatória com a origem e suas sugestões enriquecidas com `PRODUCT_NAME` e atributos-chave para revisão humana.

In [0]:
t0 = time.time()
recs = recommend(df_norm, embeddings, config, quantity_ratio_bounds=config.quantity_ratio_bounds)
t_rec = time.time() - t0

n_sug = recs["sugestoes"].apply(len)
n_total = len(recs)
n_sem = int((n_sug == 0).sum())

print(f"recommend          : {t_rec:.2f}s")
print(f"EANs               : {n_total}")
print(f"Com sugestão       : {n_total - n_sem}  ({100*(n_total-n_sem)/max(n_total,1):.1f}%)")
print(f"Sem sugestão       : {n_sem}  ({100*n_sem/max(n_total,1):.1f}%)")
print(f"Sugestões/EAN      : média={n_sug.mean():.2f}  mediana={n_sug.median():.0f}  máx={n_sug.max()}")

# Amostra qualitativa
SAMPLE_SIZE = 8
display_attrs = [r.attribute for r in config.rules if r.attribute in df_norm.columns][:5]
lookup_cols = ["PRODUCT_NAME"] + display_attrs
lookup = df_norm.set_index("EAN")[lookup_cols].to_dict(orient="index")


def explore_ean(ean: str) -> pd.DataFrame:
    row = recs[recs["ean_origem"] == ean]
    if row.empty:
        return pd.DataFrame()
    sugs = row.iloc[0]["sugestoes"]
    orig = lookup.get(ean, {})
    rows = [{"role": "ORIGEM", "rank": 0, "ean": ean, "relevance": None, **orig}]
    for s in sugs:
        info = lookup.get(s["ean"], {})
        rows.append({"role": "sugestao", "rank": s["rank"], "ean": s["ean"], "relevance": s["relevance"], **info})
    return pd.DataFrame(rows)


rng = np.random.default_rng(42)
eans_com = recs.loc[n_sug > 0, "ean_origem"].tolist()
if eans_com:
    picks = rng.choice(eans_com, size=min(SAMPLE_SIZE, len(eans_com)), replace=False)
    blocks = []
    for e in picks:
        b = explore_ean(e)
        b.insert(0, "_grupo", e)
        blocks.append(b)
    sample_view = pd.concat(blocks, ignore_index=True)
    print(sample_view.to_string())
else:
    print("Nenhum EAN com sugestões — relaxe min_score ou inspecione a config.")

---
## 5. Gravar em `groceries_ops.similis.recommendations` (+ `recommendations_flat`)

`write_recommendations` (em `parts.writer`) grava a partição `(date=hoje, subcategoria=<slug>)` na Delta. Usa `partitionOverwriteMode=dynamic`, ou seja, sobrescreve **apenas** essa partição — o resto da tabela fica intacto.

Schema da tabela aninhada:

```
ean_origem            STRING NOT NULL
product_name_origem   STRING
subcategoria          STRING NOT NULL
sugestoes             STRING            -- JSON: [{ean,relevance,rank,product_name}, ...]
n_sugestoes           INT
model_version         STRING
config_hash           STRING
date                  STRING
PARTITIONED BY (date, subcategoria)
```

Além dela, o writer grava automaticamente **`groceries_ops.similis.recommendations_flat`** — o "explode" do JSON, para averiguação pelo time de negócios. Cada linha é um par (EAN origem, EAN substituto) em ordem de rank:

```
ean_origem             STRING
product_name_origem    STRING
ean_sugestao           STRING
product_name_sugestao  STRING
rank                   INT
relevance              DOUBLE
n_sugestoes            INT      -- total de sugestões do EAN origem
subcategoria           STRING   -- slug legado da subcategoria (ex.: creme_assaduras)
categoria              STRING   -- slug da categoria (ex.: troca_de_fralda)
model_version          STRING
config_hash            STRING
date                   STRING
PARTITIONED BY (date, subcategoria, categoria)
```

> EANs **sem nenhuma sugestão** não geram linha na flat (não há par); eles continuam auditáveis na tabela aninhada com `n_sugestoes = 0`.

In [ ]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

name_by_ean = df.set_index("EAN")["PRODUCT_NAME"].to_dict()
write_recommendations(spark, recs, SUBCATEGORY_SLUG, config.config_hash, name_by_ean=name_by_ean, category_name=config.category_name, subcategoria_nested=nested_subcategoria(config.subcategory_name))

# Ambas as tabelas gravam o slug LEGADO em `subcategoria`; a flat adiciona a
# coluna `categoria` (slug) — o par (subcategoria, categoria) desambigua.
NESTED_SLUG = nested_subcategoria(config.subcategory_name)
CATEGORIA_SLUG = category_slug(config.category_name)

print("\nResumo das partições da subcategoria após o write:")
spark.sql(
    f"""
    SELECT date, subcategoria, model_version, config_hash, COUNT(*) AS n_eans
    FROM groceries_ops.similis.recommendations
    WHERE subcategoria = '{NESTED_SLUG}'
    GROUP BY date, subcategoria, model_version, config_hash
    ORDER BY date DESC
    """
).show(truncate=False)

print("\n3 primeiras linhas da partição mais recente (preview do JSON `sugestoes`):")
spark.sql(
    f"""
    SELECT ean_origem, n_sugestoes, substr(sugestoes, 1, 200) AS sugestoes_preview
    FROM groceries_ops.similis.recommendations
    WHERE subcategoria = '{NESTED_SLUG}'
      AND date = (
          SELECT MAX(date) FROM groceries_ops.similis.recommendations
          WHERE subcategoria = '{NESTED_SLUG}'
      )
    LIMIT 3
    """
).show(truncate=False)

print("\n5 primeiras linhas da flat (1 linha por par origem -> substituto):")
spark.sql(
    f"""
    SELECT ean_origem, product_name_origem, rank, relevance,
           ean_sugestao, product_name_sugestao, subcategoria, categoria
    FROM groceries_ops.similis.recommendations_flat
    WHERE subcategoria = '{NESTED_SLUG}' AND categoria = '{CATEGORIA_SLUG}'
      AND date = (
          SELECT MAX(date) FROM groceries_ops.similis.recommendations_flat
          WHERE subcategoria = '{NESTED_SLUG}' AND categoria = '{CATEGORIA_SLUG}'
      )
    ORDER BY ean_origem, rank
    LIMIT 5
    """
).show(truncate=False)


In [0]:
%sql
-- Preview limitado: a tabela cresce 1 partição por (date, subcategoria);
-- sem LIMIT o display materializa a tabela inteira no driver.
SELECT *
FROM groceries_ops.similis.recommendations
WHERE date = (SELECT MAX(date) FROM groceries_ops.similis.recommendations)
LIMIT 100;


---
## 6. (Opcional) Loop por várias subcategorias

Equivalente interativo ao `databricks bundle run similis_farma --params subcategories='[...]'`. Cada iteração grava sua própria partição em `groceries_ops.similis.recommendations`, então é seguro rodar parcialmente. Subcategorias em `EXCLUDED_SUBCATEGORIES` (i.e. `medicamentos`) são puladas.

In [0]:
SUBCATEGORIES_TO_RUN = ["lencos_umedecidos__troca_de_fralda", "talco__troca_de_fralda", "absorventes_para_seios__amamentacao", "absorvente_com_abas__absorvente_noturno", "absorvente_com_abas__absorvente_diurno", "toucas__banho", "absorvente_interno__absorvente_diurno", "sabonetes__banho", "kits_para_viagem__banho", "creme_para_assaduras__troca_de_fralda", "dispenser__papel_higienico", "papinha__alimentacao_infantil", "formula_infantil__alimentacao_infantil", "leite_em_po__alimentacao_infantil", "rolos__papel_higienico", "lencos__papel_higienico", "buchas_e_esponjas__banho" ]
# list(FARMA_SUBCATEGORIES_DEFAULT)  # ou ["fraldas", "lencos_umedecidos"]
embedder_loop = embedder if "embedder" in dir() else BgeM3Embedder()

results = []
for slug in SUBCATEGORIES_TO_RUN:
    if slug.lower() in EXCLUDED_SUBCATEGORIES:
        print(f"SKIP {slug!r}: subcategoria excluída do Similis Farma.")
        results.append({"slug": slug, "status": "excluded"})
        continue

    print(f"\n{'='*80}\n  {slug}\n{'='*80}")
    try:
        cfg = loader.get(slug)
        df_loop = load_catalog(spark, cfg.subcategory_name, cfg)
        if df_loop.empty:
            print("  catálogo vazio — pulando.")
            results.append({"slug": slug, "status": "empty", "subcategory_name": cfg.subcategory_name})
            continue

        df_n = normalize_dataframe(df_loop, cfg)
        embs = embedder_loop.encode_dataframe(df_n, slug)
        recs_l = recommend(df_n, embs, cfg)
        name_by_ean = df_n.set_index("EAN")["PRODUCT_NAME"].to_dict()
        write_recommendations(spark, recs_l, slug, cfg.config_hash, name_by_ean=name_by_ean)

        n_sug_l = recs_l["sugestoes"].apply(len)
        results.append({
            "slug": slug,
            "status": "ok",
            "subcategory_name": cfg.subcategory_name,
            "config_source": cfg.source,
            "n_eans": len(df_loop),
            "com_sugestao": int((n_sug_l > 0).sum()),
            "sem_sugestao": int((n_sug_l == 0).sum()),
            "config_hash": cfg.config_hash,
        })
        print(f"  OK — {results[-1]['com_sugestao']}/{results[-1]['n_eans']} EANs com sugestão.")
    except Exception as e:
        print(f"  ERRO em {slug}: {e}")
        results.append({"slug": slug, "status": "error", "error": str(e)})

display(pd.DataFrame(results))

---
## 7. Diagnóstico — distribuição de sugestões e relevâncias

Lê a **partição mais recente de cada subcategoria** nas duas tabelas e gera:

1. Boxplot do nº de sugestões por EAN, por subcategoria (tabela aninhada — inclui EANs com 0 sugestões).
2. Distribuição das relevâncias por subcategoria (tabela flat): estatísticas descritivas, boxplot e decaimento da relevância mediana por rank.

Útil para detectar subcategorias com `min_score` agressivo demais (caudas de 0 sugestões), partições de `hard_filter` esvaziadas, ou relevâncias "achatadas" perto do corte.

In [ ]:
import matplotlib.pyplot as plt

# ---- nº de sugestões por EAN (tabela aninhada; partição mais recente por subcategoria) ----
nsug_pdf = spark.sql(
    """
    WITH latest AS (
        SELECT subcategoria, MAX(date) AS date
        FROM groceries_ops.similis.recommendations
        GROUP BY subcategoria
    )
    SELECT r.subcategoria, r.date, r.ean_origem, r.n_sugestoes
    FROM groceries_ops.similis.recommendations r
    JOIN latest l
      ON r.subcategoria = l.subcategoria AND r.date = l.date
    """
).toPandas()

print("Partições usadas (mais recente por subcategoria):")
print(nsug_pdf.groupby(["subcategoria", "date"]).size().rename("n_eans").to_string())

resumo_nsug = (
    nsug_pdf.groupby("subcategoria")["n_sugestoes"]
    .agg(
        n_eans="count",
        pct_sem_sugestao=lambda s: 100.0 * (s == 0).mean(),
        media="mean",
        p25=lambda s: s.quantile(0.25),
        mediana="median",
        p75=lambda s: s.quantile(0.75),
        maximo="max",
    )
    .round(2)
    .sort_values("mediana", ascending=False)
)
print("\nResumo de n_sugestoes por subcategoria:")
print(resumo_nsug.to_string())

# ---- boxplot ----
ordem = resumo_nsug.index.tolist()
dados = [nsug_pdf.loc[nsug_pdf["subcategoria"] == s, "n_sugestoes"] for s in ordem]

fig, ax = plt.subplots(figsize=(max(6, 1.6 * len(ordem)), 5))
bp = ax.boxplot(dados, labels=ordem, showmeans=True, patch_artist=True)
for box in bp["boxes"]:
    box.set(facecolor="#EA1D2C", alpha=0.35)
ax.set_title("Nº de sugestões por EAN, por subcategoria (partição mais recente)")
ax.set_ylabel("n_sugestoes")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# ---- distribuição das relevâncias (tabela flat; partição mais recente por subcategoria) ----
rel_pdf = spark.sql(
    """
    WITH latest AS (
        SELECT subcategoria, MAX(date) AS date
        FROM groceries_ops.similis.recommendations_flat
        GROUP BY subcategoria
    )
    SELECT f.subcategoria, f.ean_origem, f.rank, f.relevance
    FROM groceries_ops.similis.recommendations_flat f
    JOIN latest l
      ON f.subcategoria = l.subcategoria AND f.date = l.date
    """
).toPandas()

resumo_rel = (
    rel_pdf.groupby("subcategoria")["relevance"]
    .agg(
        n_pares="count",
        media="mean",
        desvio="std",
        minimo="min",
        p10=lambda s: s.quantile(0.10),
        p25=lambda s: s.quantile(0.25),
        mediana="median",
        p75=lambda s: s.quantile(0.75),
        p90=lambda s: s.quantile(0.90),
        maximo="max",
    )
    .round(4)
)
# relevância do melhor substituto (rank 1) por subcategoria — termômetro de qualidade do "top pick"
top1 = (
    rel_pdf[rel_pdf["rank"] == 1]
    .groupby("subcategoria")["relevance"]
    .agg(top1_mediana="median", top1_p10=lambda s: s.quantile(0.10))
    .round(4)
)
print("Distribuição das relevâncias por subcategoria (todos os pares):")
print(resumo_rel.join(top1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(max(12, 3.2 * rel_pdf["subcategoria"].nunique()), 5))

# (a) boxplot da relevância por subcategoria
ordem_rel = resumo_rel.sort_values("mediana", ascending=False).index.tolist()
dados_rel = [rel_pdf.loc[rel_pdf["subcategoria"] == s, "relevance"] for s in ordem_rel]
bp = axes[0].boxplot(dados_rel, labels=ordem_rel, showmeans=True, patch_artist=True)
for box in bp["boxes"]:
    box.set(facecolor="#1f77b4", alpha=0.35)
axes[0].set_title("Relevância por subcategoria")
axes[0].set_ylabel("relevance")
axes[0].grid(axis="y", alpha=0.3)
axes[0].tick_params(axis="x", rotation=20)

# (b) decaimento: relevância mediana por rank (até rank 20), por subcategoria
for s in ordem_rel:
    sub = rel_pdf[(rel_pdf["subcategoria"] == s) & (rel_pdf["rank"] <= 20)]
    curva = sub.groupby("rank")["relevance"].median()
    axes[1].plot(curva.index, curva.values, marker="o", ms=3, label=s)
axes[1].set_title("Relevância mediana por rank (até 20)")
axes[1].set_xlabel("rank")
axes[1].set_ylabel("relevance mediana")
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
#  BASELINE — 100 EANs por subcategoria, só as ainda NÃO rodadas, em tabelas _temp
#  Self-contained: NÃO importa parts.writer nem parts.constants. Define a própria
#  gravação e TRAVA qualquer destino que não termine em "_temp".
#  Depende só das células de import já executadas: ConfigLoader (loader),
#  load_catalog, normalize_dataframe, recommend, BgeM3Embedder,
#  EXCLUDED_SUBCATEGORIES.
# ============================================================================
import json
from datetime import datetime

from pyspark.sql.types import (
    DoubleType, IntegerType, StringType, StructField, StructType,
)

# --- parâmetros --------------------------------------------------------------
MAX_EANS        = 100
TEMP_TABLE      = "groceries_ops.similis.recommendations_temp"
TEMP_FLAT_TABLE = "groceries_ops.similis.recommendations_flat_temp"
MODEL_VERSION   = "bge-m3-v1.4"   # literal (era a constante; aqui sem dependência)

# >>> TRAVA: impossível por construção gravar fora de _temp <<<
assert TEMP_TABLE.endswith("_temp") and TEMP_FLAT_TABLE.endswith("_temp"), (
    "TRAVA: destino não termina em '_temp' — abortado para não tocar produção."
)

JA_RODADAS = {
    "fraldas", "lencos_umedecidos", "formula_infantil", "papinha", "sabonetes",
    "talco", "buchas_e_esponjas", "creme_assaduras", "rolos", "toucas",
    "dispenser", "leite_em_po", "lencos", "kits_para_viagem",
}

_NESTED_SCHEMA = StructType([
    StructField("ean_origem",          StringType(),  True),
    StructField("product_name_origem", StringType(),  True),
    StructField("subcategoria",        StringType(),  True),
    StructField("sugestoes",           StringType(),  True),
    StructField("n_sugestoes",         IntegerType(), True),
    StructField("model_version",       StringType(),  True),
    StructField("config_hash",         StringType(),  True),
    StructField("date",                StringType(),  True),
])
_FLAT_SCHEMA = StructType([
    StructField("ean_origem",            StringType(),  True),
    StructField("product_name_origem",   StringType(),  True),
    StructField("ean_sugestao",          StringType(),  True),
    StructField("product_name_sugestao", StringType(),  True),
    StructField("rank",                  IntegerType(), True),
    StructField("relevance",             DoubleType(),  True),
    StructField("n_sugestoes",           IntegerType(), True),
    StructField("subcategoria",          StringType(),  True),
    StructField("model_version",         StringType(),  True),
    StructField("config_hash",           StringType(),  True),
    StructField("date",                  StringType(),  True),
])


def _as_list(sugs):
    if sugs is None:
        return []
    if isinstance(sugs, str):
        try:
            sugs = json.loads(sugs)
        except (TypeError, ValueError):
            return []
    return list(sugs) if isinstance(sugs, (list, tuple)) else []


def _write_temp(spark, recs_df, subcategoria, config_hash, name_by_ean,
                table=TEMP_TABLE, flat_table=TEMP_FLAT_TABLE):
    """Grava recs nas tabelas _temp. Espelha o writer de producao (mesmas colunas
    e particionamento), mas com destino fixo em _temp e trava de seguranca."""
    # trava tambem em runtime: nenhuma gravacao escapa para fora de _temp
    assert table.endswith("_temp") and flat_table.endswith("_temp"), (
        f"TRAVA: destino {table!r}/{flat_table!r} nao e _temp — gravacao abortada."
    )
    if recs_df.empty:
        print(f"  nada a gravar ({subcategoria}).")
        return

    run_date = datetime.today().strftime("%Y-%m-%d")
    name_by_ean = name_by_ean or {}
    out = recs_df.copy()
    out["subcategoria"] = subcategoria
    out["n_sugestoes"]  = out["sugestoes"].apply(lambda s: len(_as_list(s)))

    def _enrich(sugs):
        return [
            {**s, "product_name": s.get("product_name") or name_by_ean.get(s.get("ean"))}
            for s in _as_list(sugs)
        ]

    out["sugestoes"]           = out["sugestoes"].apply(_enrich)
    out["product_name_origem"] = out["ean_origem"].map(lambda e: name_by_ean.get(e))
    out["model_version"]       = MODEL_VERSION
    out["config_hash"]         = config_hash
    out["date"]                = run_date

    # flat (1 linha por par) — construida ANTES da serializacao JSON
    flat_rows = []
    for rec in out.itertuples(index=False):
        for s in _as_list(rec.sugestoes):
            flat_rows.append({
                "ean_origem":            rec.ean_origem,
                "product_name_origem":   rec.product_name_origem,
                "ean_sugestao":          s.get("ean"),
                "product_name_sugestao": s.get("product_name"),
                "rank":      int(s["rank"]) if s.get("rank") is not None else None,
                "relevance": float(s["relevance"]) if s.get("relevance") is not None else None,
                "n_sugestoes":   rec.n_sugestoes,
                "subcategoria":  rec.subcategoria,
                "model_version": rec.model_version,
                "config_hash":   rec.config_hash,
                "date":          rec.date,
            })
    flat = pd.DataFrame(flat_rows, columns=[f.name for f in _FLAT_SCHEMA.fields])
    if not flat.empty:
        flat = flat.sort_values(["ean_origem", "rank"]).reset_index(drop=True)

    # serializa sugestoes -> JSON e grava a aninhada
    out["sugestoes"] = out["sugestoes"].apply(lambda x: json.dumps(x, ensure_ascii=False))
    out = out[[f.name for f in _NESTED_SCHEMA.fields]]
    (spark.createDataFrame(out, schema=_NESTED_SCHEMA)
        .write.mode("overwrite")
        .option("partitionOverwriteMode", "dynamic")
        .option("mergeSchema", "true")
        .partitionBy("date", "subcategoria")
        .saveAsTable(table))
    print(f"  Gravado {len(out)} linhas em {table} (date={run_date}, subcat={subcategoria})")

    if not flat.empty:
        flat = flat[[f.name for f in _FLAT_SCHEMA.fields]]
        (spark.createDataFrame(flat, schema=_FLAT_SCHEMA)
            .write.mode("overwrite")
            .option("partitionOverwriteMode", "dynamic")
            .option("mergeSchema", "true")
            .partitionBy("date", "subcategoria")
            .saveAsTable(flat_table))
        print(f"  Gravado {len(flat)} pares em {flat_table}")


# --- subcategorias a rodar = todas do infos.yaml - ja rodadas - excluidas ----
todos_slugs = list(loader._list_ids.keys())
SUBCATEGORIES_TO_RUN = [
    s for s in todos_slugs
    if s.lower() not in JA_RODADAS and s.lower() not in EXCLUDED_SUBCATEGORIES
]
print(f"Total no infos.yaml : {len(todos_slugs)}")
print(f"Ja rodadas          : {len(JA_RODADAS)}")
print(f"A rodar (baseline)  : {len(SUBCATEGORIES_TO_RUN)}")
print(f"Destino             : {TEMP_TABLE}")
print(f"                      {TEMP_FLAT_TABLE}")

embedder_loop = embedder if "embedder" in dir() else BgeM3Embedder()
results = []
for i, slug in enumerate(SUBCATEGORIES_TO_RUN, 1):
    print(f"\n{'='*80}\n  [{i}/{len(SUBCATEGORIES_TO_RUN)}] {slug}\n{'='*80}")
    try:
        cfg = loader.get(slug)

        spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")  # toPandas
        df_loop = load_catalog(spark, cfg.subcategory_name, cfg)
        if df_loop.empty:
            print("  catalogo vazio — pulando.")
            results.append({"slug": slug, "status": "empty",
                            "subcategory_name": cfg.subcategory_name})
            continue

        # corte deterministico de 100 EANs (troque por
        # .sample(n=MAX_EANS, random_state=42) se quiser amostra aleatoria)
        n_full = len(df_loop)
        if n_full > MAX_EANS:
            df_loop = df_loop.head(MAX_EANS).reset_index(drop=True)
        print(f"  {len(df_loop)}/{n_full} EANs (cap {MAX_EANS})")

        df_n  = normalize_dataframe(df_loop, cfg)
        embs  = embedder_loop.encode_dataframe(df_n, slug)
        recs_l = recommend(df_n, embs, cfg,
                           quantity_ratio_bounds=cfg.quantity_ratio_bounds)
        name_by_ean = df_n.set_index("EAN")["PRODUCT_NAME"].to_dict()

        spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")   # createDataFrame
        _write_temp(spark, recs_l, slug, cfg.config_hash, name_by_ean)

        n_sug_l = recs_l["sugestoes"].apply(len)
        results.append({
            "slug": slug, "status": "ok",
            "subcategory_name": cfg.subcategory_name,
            "n_eans": len(df_loop),
            "com_sugestao": int((n_sug_l > 0).sum()),
            "sem_sugestao": int((n_sug_l == 0).sum()),
            "config_hash": cfg.config_hash,
        })
        print(f"  OK — {results[-1]['com_sugestao']}/{results[-1]['n_eans']} com sugestao.")
    except Exception as e:
        print(f"  ERRO em {slug}: {e}")
        results.append({"slug": slug, "status": "error", "error": str(e)})

resumo = pd.DataFrame(results)
print("\n" + "="*80 + "\n  RESUMO\n" + "="*80)
print(resumo["status"].value_counts().to_string())
display(resumo)